# Code 8.6.3: In-context learning with GPT-2

Measures the accuracy of GPT-2 small and medium on the first 100 SST-2 validation reviews with 0, 1, 4, and 8 labeled training reviews in the prompt, by comparing the next-token logits of " negative" and " positive". Accuracies for $`k \ge 1`$ are averaged over three random draws of examples. Figure 8.6.1 plots these numbers.


In [ ]:
import random
import torch
from datasets import load_dataset
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

sst2 = load_dataset("stanfordnlp/sst2")
train = [ex for ex in sst2["train"] if 8 <= len(ex["sentence"].split()) <= 25]   # short, complete reviews
test = list(sst2["validation"])[:100]
words = {0: " negative", 1: " positive"}

def prompt(demos, sentence):
    shots = "".join(f"Review: {d['sentence'].strip()}\nSentiment:{words[d['label']]}\n\n" for d in demos)
    return shots + f"Review: {sentence.strip()}\nSentiment:"

@torch.no_grad()
def accuracy(model, tok, k, seed):
    rng = random.Random(seed)
    label_ids = [tok.encode(words[0])[0], tok.encode(words[1])[0]]
    correct = 0
    for ex in test:
        demos = rng.sample(train, k)                    # k random labeled examples in the prompt
        ids = tok(prompt(demos, ex["sentence"]), return_tensors="pt").input_ids
        logits = model(ids).logits[0, -1, label_ids]    # compare the two label words only
        correct += int(logits.argmax()) == ex["label"]
    return correct / len(test)

print("majority class:", max(sum(ex["label"] for ex in test), sum(1 - ex["label"] for ex in test)) / len(test))
for name in ["gpt2", "gpt2-medium"]:
    tok = GPT2TokenizerFast.from_pretrained(name)
    model = GPT2LMHeadModel.from_pretrained(name).eval()
    row = []
    for k in [0, 1, 4, 8]:
        accs = [accuracy(model, tok, k, seed) for seed in ([0] if k == 0 else [0, 1, 2])]
        row.append(f"k={k}: {sum(accs) / len(accs):.3f}")
    print(name, ", ".join(row))
# majority class: 0.52
# gpt2 k=0: 0.610, k=1: 0.600, k=4: 0.567, k=8: 0.523
# gpt2-medium k=0: 0.770, k=1: 0.697, k=4: 0.573, k=8: 0.630
